# Engine — authority editions, pinning, and deterministic replay

**Topic:** the `Engine` — the immutable `name -> Authority` binding that selects which concrete authority *editions* Paxman uses for a canonicalize call.  
**Public API:** `paxman.Engine`, `paxman.canonicalize_with`, `paxman.Edition`, `paxman.Latest`, `paxman.replay`.

This notebook follows the **reference template** (`00_email.ipynb`). Run every cell in order; no hidden state. It builds on the capability notebooks — here we look at the *execution environment* underneath `canonicalize`, not a single data domain.

> Why this matters: Paxman is deterministic (mandate Law 1). The Engine is what makes a canonicalization **replayable** — the artifact records exactly which authority editions produced it, so `replay` reconstructs the same result byte-for-byte even if a newer edition ships later.

In [ ]:
from paxman import canonicalize, canonicalize_with, replay, Engine, Edition, Latest
from paxman._errors import UnknownAuthorityEdition

def show(raw, contract, engine=None):
    """Canonicalize with an optional engine; print status + value."""
    art = canonicalize_with(raw, contract, engine) if engine else canonicalize(raw, contract)
    print(f"engine={'pinned' if engine else 'default':7} {raw!r:22} -> {art.status.name:14} {art.value!r}")
    return art

## `Engine.default()` — the zero-config path

`paxman.canonicalize(...)` is just `canonicalize_with(..., Engine.default())`. The default engine binds **every** known authority to its active (latest) edition. That's 18 authorities in this build.

In [ ]:
from paxman import Email

e0 = Engine.default()
print("bound authorities:", len(e0.authorities()))
# One concrete edition, as recorded on the artifact:
print("sample:", e0.authorities()[0])

# canonicalize() and canonicalize_with(..., Engine.default()) are equivalent:
show("A@B.COM", Email())
show("A@B.COM", Email(), e0)

## `Engine.with_authorities(...)` — pin specific editions

Pin one or more authorities to a concrete `Edition(id)` or `Latest()`. This is how an organization locks a *compliance profile* so artifacts are replay-deterministic against adopted editions, independent of what Paxman later bundles.

Note: with a single bundled edition today, pinning `ISO 3166-1` to `"2024"` does not change an **email** result (email depends on RFC 5321 + the paxman email spec, not ISO 3166). The point of pinning is *future-proofing*: when a newer edition ships, already-produced artifacts still replay against the edition that made them.

In [ ]:
e_pin = Engine.with_authorities({"ISO 3166-1": Edition("2024")})
print("pinned ISO 3166-1 edition:", e_pin.authority("ISO 3166-1").edition)

show("A@B.COM", Email(), e_pin)   # same email output, explicit edition context

# Edition vs Latest:
e_latest = Engine.with_authorities({"ISO 3166-1": Latest()})
print("Latest() resolves to:", e_latest.authority("ISO 3166-1").edition)

## Bad pins raise `UnknownAuthorityEdition`

Only authorities Paxman actually bundles are accepted, and only known edition ids. A bad authority name or an unknown edition id fails **at engine construction**, before any canonicalize runs.

In [ ]:
for name, sel in [("BOGUS", Latest()), ("ISO 3166-1", Edition("1999"))]:
    try:
        Engine.with_authorities({name: sel})
        print("unexpectedly OK:", name)
    except UnknownAuthorityEdition as exc:
        print(f"UnknownAuthorityEdition ({name}): {str(exc)[:60]}...")

## `authority_override` — the per-contract escape hatch

A contract can pin one authority for a single call, layered on top of the engine. Useful for testing a specific edition without building a whole engine.

In [ ]:
c = Email(authority_override={"ISO 3166-1": Latest()})
art = canonicalize("A@B.COM", c)
print("override contract ->", art.status.name, art.value)
print("authorities recorded on artifact:", [a.name for a in art.authorities])

## `replay` — byte-equal rehydration (the payoff)

`replay(artifact, contract)` rebuilds the exact production context from the editions recorded on the artifact (`Engine.from_artifact`). The result is byte-for-byte equal to the original — this is what makes pinned editions durable.

In [ ]:
art = canonicalize("John.Doe+spam@gmail.COM", Email(provider_aliases="gmail"))
art2 = replay(art, Email(provider_aliases="gmail"))
print("original :", art.status.name, art.value)
print("replayed :", art2.status.name, art2.value)
print("byte-equal value:", art.value == art2.value)

# The engine reconstructed from the artifact:
e_rec = Engine.from_artifact(art.authorities)
print("reconstructed authorities:", len(e_rec.authorities()))

## Where to go next

- **`11_dsl.ipynb`** — build contracts from a DSL string with `parse_contract` (no need to import each value object).
- Capability notebooks `00_email.ipynb` … `09_uuid.ipynb` — the data domains this engine canonicalizes.
- Background: `NOTEBOOK_INPUTS.md` (verified against the working tree) and `ARCHITECTURE.md` (three-layer authority model).